> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `1005795510073716` · task `bronze_to_silver` · notebook `02_bronze_to_silver` · started 2026-11-17 20:32 UTC · result **SUCCESS**  
> Parameters: `batch_ids={{job.parameters.batch_ids}}`, `catalog={{job.parameters.catalog}}`, `end_date={{job.parameters.end_date}}`, `mode={{job.parameters.mode}}`, `start_date={{job.parameters.start_date}}`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 02 · Bronze → Silver
Casts epoch integers to timestamps, decodes titles, removes PII (drops names/avatars, hashes user ids with a
salted SHA-256), derives body features, applies data-quality rules and upserts every Silver table with
**`MERGE INTO`**. Re-running the same batches changes nothing (idempotent).

| Widget | Standard incremental run | Backfill |
|---|---|---|
| `mode` | `incremental`: Bronze batches not yet in Silver | `backfill`: re-process the selected batches |
| `batch_ids` | empty | comma-separated batch ids |
| `start_date` / `end_date` | empty | extraction-date range, `YYYY-MM-DD`, inclusive |
| `pii_salt` | empty (read from secret `so_lakehouse/pii_salt`) | same |

In [1]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("batch_ids", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("pii_salt", "")

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
from so_lakehouse.silver import run_bronze_to_silver

cfg = build_config(need_salt=True)
audit = run_bronze_to_silver(
    spark, cfg,
    mode=widget("mode"),
    batch_ids=widget("batch_ids") or None,
    start_date=widget("start_date") or None,
    end_date=widget("end_date") or None,
    fail_on_error=True,
)

Bronze-to-Silver: processing 2 batch(es): incr_20261008T000000Z, incr_20261117T202706Z
[SUCCESS] Bronze-to-Silver | questions                          | read=35 ins=32 upd=1 del=0 quar=0 | 38.6s | 35 unique questions after de-duplication
[SUCCESS] Bronze-to-Silver | question_tags                      | read=125 ins=113 upd=0 del=0 quar=0 | 26.7s
[SUCCESS] Bronze-to-Silver | answers                            | read=49 ins=49 upd=0 del=0 quar=0 | 15.0s | 49 unique answers after de-duplication
[SUCCESS] Bronze-to-Silver | users                              | read=84 ins=59 upd=19 del=0 quar=0 | 14.3s
[SUCCESS] Bronze-to-Silver | question_snapshots                 | read=0 ins=0 upd=0 del=0 quar=0 | 7.9s
[SUCCESS] Bronze-to-Silver | question_metrics (from snapshots)  | read=0 ins=0 upd=0 del=0 quar=0 | 9.7s
[SUCCESS] Bronze-to-Silver | deleted_questions -> questions     | read=0 ins=0 upd=0 del=0 quar=0 | 7.7s | soft delete (is_deleted = true)
[SUCCESS] Bronze-to-Silver | deleted_question

In [4]:
display(spark.table(cfg.table("ops.pipeline_execution_logs"))
        .where(f"run_id = '{audit.run_id}'").orderBy("start_time"))

log_id,run_id,pipeline_name,layer,run_mode,load_type,entity,batch_id,source_parameter,target_table,start_time,end_time,duration_seconds,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message,load_timestamp
4ac9a56c-33a3-4ebb-94c4-728190986ff5,bronze_to_silver-20261117T203206Z-8dd880,bronze_to_silver,Bronze-to-Silver,incremental,incremental,questions,,"unprocessed batches -> [incr_20261008T000000Z, incr_20261117T202706Z]",workspace.so_silver.questions,2026-11-17T20:32:08.824Z,2026-11-17T20:32:47.393Z,38.569467,SUCCESS,35,32,1,0,0,35 unique questions after de-duplication,2026-11-17T20:32:47.393Z
1868626d-dfe6-4da7-bc5e-c3c3d43f6abc,bronze_to_silver-20261117T203206Z-8dd880,bronze_to_silver,Bronze-to-Silver,incremental,incremental,question_tags,,"unprocessed batches -> [incr_20261008T000000Z, incr_20261117T202706Z]",workspace.so_silver.question_tags,2026-11-17T20:32:50.187Z,2026-11-17T20:33:16.889Z,26.702079,SUCCESS,125,113,0,0,0,,2026-11-17T20:33:16.889Z
8cb84c9e-8bc4-439b-bef7-dd7c84d4124c,bronze_to_silver-20261117T203206Z-8dd880,bronze_to_silver,Bronze-to-Silver,incremental,incremental,answers,,"unprocessed batches -> [incr_20261008T000000Z, incr_20261117T202706Z]",workspace.so_silver.answers,2026-11-17T20:33:19.050Z,2026-11-17T20:33:34.082Z,15.032936,SUCCESS,49,49,0,0,0,49 unique answers after de-duplication,2026-11-17T20:33:34.082Z
e41cfdf4-91aa-43f9-9ff9-a8c7d77fd5ad,bronze_to_silver-20261117T203206Z-8dd880,bronze_to_silver,Bronze-to-Silver,incremental,incremental,users,,"unprocessed batches -> [incr_20261008T000000Z, incr_20261117T202706Z]",workspace.so_silver.users,2026-11-17T20:33:36.038Z,2026-11-17T20:33:50.332Z,14.293694,SUCCESS,84,59,19,0,0,,2026-11-17T20:33:50.332Z
00011c0d-2632-4829-b1fb-36852b76b27c,bronze_to_silver-20261117T203206Z-8dd880,bronze_to_silver,Bronze-to-Silver,incremental,incremental,question_snapshots,,"unprocessed batches -> [incr_20261008T000000Z, incr_20261117T202706Z]",workspace.so_silver.question_snapshots,2026-11-17T20:33:52.373Z,2026-11-17T20:34:00.294Z,7.920889,SUCCESS,0,0,0,0,0,,2026-11-17T20:34:00.294Z
0bf8c0ba-c336-4312-ae35-dcf115c3c99d,bronze_to_silver-20261117T203206Z-8dd880,bronze_to_silver,Bronze-to-Silver,incremental,incremental,question_metrics (from snapshots),,"unprocessed batches -> [incr_20261008T000000Z, incr_20261117T202706Z]",workspace.so_silver.questions,2026-11-17T20:34:02.383Z,2026-11-17T20:34:12.072Z,9.689275,SUCCESS,0,0,0,0,0,,2026-11-17T20:34:12.072Z
f6a501ce-5225-4774-a550-a1e0b8166858,bronze_to_silver-20261117T203206Z-8dd880,bronze_to_silver,Bronze-to-Silver,incremental,incremental,deleted_questions -> questions,,"unprocessed batches -> [incr_20261008T000000Z, incr_20261117T202706Z]",workspace.so_silver.questions,2026-11-17T20:34:13.954Z,2026-11-17T20:34:21.670Z,7.715577,SUCCESS,0,0,0,0,0,soft delete (is_deleted = true),2026-11-17T20:34:21.670Z
e9fc3bac-55eb-47ab-b96d-99ac9620f672,bronze_to_silver-20261117T203206Z-8dd880,bronze_to_silver,Bronze-to-Silver,incremental,incremental,deleted_questions -> answers,,"unprocessed batches -> [incr_20261008T000000Z, incr_20261117T202706Z]",workspace.so_silver.answers,2026-11-17T20:34:24.046Z,2026-11-17T20:34:32.469Z,8.422716,SUCCESS,0,0,0,0,0,soft delete (is_deleted = true),2026-11-17T20:34:32.469Z
d0f4392f-7fac-46a6-b3e5-92f59630134b,bronze_to_silver-20261117T203206Z-8dd880,bronze_to_silver,Bronze-to-Silver,incremental,incremental,post_text_masked,,"unprocessed batches -> [incr_20261008T000000Z, incr_20261117T202706Z]",workspace.so_silver.post_text_masked,2026-11-17T20:34:34.697Z,2026-11-17T20:34:53.127Z,18.430094,SUCCESS,84,81,0,0,0,,2026-11-17T20:34:53.127Z
